<a href="https://colab.research.google.com/github/stauntonjr/local_llm_notebooks/blob/master/Optimizing_Temperature%2C_Top_P%2C_and_Min_P_with_vLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*More details in this article: [Using GGUF Models? Optimize Your Inference Temperature!](https://kaitchup.substack.com/p/using-gguf-models-optimize-your-inference)*


The following notebook show how to evaluate different versions of Qwen3 models (here, the 8B size) with different combination of inference hyperparameters. It uses vLLM and lm-eval to efficiently get the results.

# Installation

In [ ]:
!git clone --depth 1 https://github.com/EleutherAI/lm-evaluation-harness && cd lm-evaluation-harness && pip install -e .
!pip install --upgrade vllm

# Hyperparameter Search

Note 1: at t=0.0, p (top_p) and mp (min_p) shouldn't have any impact on the results

Note 2: And yes, the V1 engine is still not stable enough to smoothly run all these variants. I recommend disabling it with VLLM_USE_V1=0.

In [ ]:
%%bash
for t in 0.0 0.2 0.4 0.6 0.8 1.0;
do
    for p in  1.0 0.9 0.8;
    do
        for mp in 0.0 0.5 0.1 .2  ;
        do

              VLLM_USE_V1=0 lm_eval --model vllm \
              --model_args pretrained="Qwen/Qwen3-8B",tokenizer="Qwen/Qwen3-8B",dtype="bfloat16",max_model_len=12000,enable_thinking=False \
              --tasks leaderboard_ifeval \
              --device cuda:0 \
              --gen_kwargs temperature=${t},top_p=${p},min_p=${mp} \
              --batch_size auto \
              --apply_chat_template \
              --output_path results_nothinking/${t}-${p}-${mp}

              VLLM_USE_V1=0 lm_eval --model vllm \
              --model_args pretrained="unsloth/Qwen3-8B-bnb-4bit",load_format="bitsandbytes",tokenizer="Qwen/Qwen3-8B",dtype="bfloat16",max_model_len=12000,enable_thinking=False \
              --tasks leaderboard_ifeval \
              --device cuda:0 \
              --gen_kwargs temperature=${t},top_p=${p},min_p=${mp} \
              --batch_size auto \
              --apply_chat_template \
              --output_path results_nothinking/${t}-${p}-${mp}

            VLLM_USE_V1=0 lm_eval --model vllm \
              --model_args pretrained="Qwen/Qwen3-8B-AWQ",tokenizer="Qwen/Qwen3-8B",dtype="bfloat16",max_model_len=12000,enable_thinking=False \
              --tasks leaderboard_ifeval \
              --device cuda:0 \
              --gen_kwargs temperature=${t},top_p=${p},min_p=${mp} \
              --batch_size auto \
              --apply_chat_template \
              --output_path results_nothinking/${t}-${p}-${mp}

              VLLM_USE_V1=0 lm_eval --model vllm \
              --model_args pretrained="Qwen3-8B-Q2_K.gguf",load_format="gguf",trust_remote_code=True,tokenizer="Qwen/Qwen3-8B",dtype="bfloat16",max_model_len=12000,enable_thinking=False \
              --tasks leaderboard_ifeval \
              --device cuda:0 \
              --gen_kwargs temperature=${t},top_p=${p},min_p=${mp} \
              --batch_size auto \
              --apply_chat_template \
              --output_path results_nothinking/${t}-${p}-${mp}

              VLLM_USE_V1=0 lm_eval --model vllm \
              --model_args pretrained="Qwen3-8B-Q4_K_M.gguf",tokenizer="Qwen/Qwen3-8B",load_format="gguf",dtype="bfloat16",max_model_len=12000,enable_thinking=False \
              --tasks leaderboard_ifeval \
              --device cuda:0 \
              --gen_kwargs temperature=${t},top_p=${p},min_p=${mp} \
              --batch_size auto \
              --apply_chat_template \
              --output_path results_nothinking/${t}-${p}-${mp}
        done
    done
done

# Results (CSV format)

In [ ]:
temperature,top_p,min_p,model,metric,value
0.2,0.8,0.0,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8170055452865065
0.2,0.8,0.0,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8133086876155268
0.2,0.8,0.0,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8151571164510166
0.2,0.8,0.0,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8059149722735675
0.2,0.8,0.0,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8096118299445472
0.2,0.8,0.05,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8170055452865065
0.2,0.8,0.05,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8133086876155268
0.2,0.8,0.05,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8133086876155268
0.2,0.8,0.05,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8059149722735675
0.2,0.8,0.05,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8096118299445472
0.2,0.8,0.1,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8170055452865065
0.2,0.8,0.1,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8133086876155268
0.2,0.8,0.1,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8133086876155268
0.2,0.8,0.1,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8059149722735675
0.2,0.8,0.1,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8096118299445472
0.2,0.8,0.2,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8262476894639557
0.2,0.8,0.2,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8022181146025879
0.2,0.8,0.2,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8022181146025879
0.2,0.8,0.2,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.800369685767098
0.2,0.8,0.2,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8133086876155268
0.2,0.9,0.0,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8170055452865065
0.2,0.9,0.0,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.800369685767098
0.2,0.9,0.0,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8151571164510166
0.2,0.9,0.0,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.800369685767098
0.2,0.9,0.0,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.822550831792976
0.2,0.9,0.05,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8170055452865065
0.2,0.9,0.05,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8096118299445472
0.2,0.9,0.05,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8096118299445472
0.2,0.9,0.05,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8040665434380776
0.2,0.9,0.05,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8207024029574861
0.2,0.9,0.1,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8151571164510166
0.2,0.9,0.1,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8059149722735675
0.2,0.9,0.1,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8207024029574861
0.2,0.9,0.1,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7966728280961183
0.2,0.9,0.1,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8188539741219963
0.2,0.9,0.2,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8280961182994455
0.2,0.9,0.2,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8059149722735675
0.2,0.9,0.2,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8170055452865065
0.2,0.9,0.2,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7929759704251387
0.2,0.9,0.2,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8280961182994455
0.2,1.0,0.0,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8243992606284658
0.2,1.0,0.0,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8096118299445472
0.2,1.0,0.0,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8133086876155268
0.2,1.0,0.0,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7966728280961183
0.2,1.0,0.0,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8428835489833642
0.2,1.0,0.05,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8151571164510166
0.2,1.0,0.05,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8114602587800369
0.2,1.0,0.05,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8096118299445472
0.2,1.0,0.05,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.800369685767098
0.2,1.0,0.05,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.833641404805915
0.2,1.0,0.1,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8133086876155268
0.2,1.0,0.1,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8059149722735675
0.2,1.0,0.1,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8133086876155268
0.2,1.0,0.1,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7985212569316081
0.2,1.0,0.1,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8373382624768947
0.2,1.0,0.2,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8096118299445472
0.2,1.0,0.2,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.800369685767098
0.2,1.0,0.2,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8170055452865065
0.2,1.0,0.2,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7911275415896488
0.2,1.0,0.2,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8280961182994455
0.4,0.8,0.0,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8188539741219963
0.4,0.8,0.0,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8299445471349353
0.4,0.8,0.0,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8059149722735675
0.4,0.8,0.0,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7985212569316081
0.4,0.8,0.0,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8114602587800369
0.4,0.8,0.05,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8188539741219963
0.4,0.8,0.05,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8280961182994455
0.4,0.8,0.05,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8059149722735675
0.4,0.8,0.05,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.800369685767098
0.4,0.8,0.05,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8114602587800369
0.4,0.8,0.1,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8207024029574861
0.4,0.8,0.1,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8151571164510166
0.4,0.8,0.1,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8096118299445472
0.4,0.8,0.1,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7985212569316081
0.4,0.8,0.1,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8133086876155268
0.4,0.8,0.2,Qwen__Qwen3-8B,prompt_level_strict_acc,0.844731977818854
0.4,0.8,0.2,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8151571164510166
0.4,0.8,0.2,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.800369685767098
0.4,0.8,0.2,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7985212569316081
0.4,0.8,0.2,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8170055452865065
0.4,0.9,0.0,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8243992606284658
0.4,0.9,0.0,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8133086876155268
0.4,0.9,0.0,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8040665434380776
0.4,0.9,0.0,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8040665434380776
0.4,0.9,0.0,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8317929759704251
0.4,0.9,0.05,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8280961182994455
0.4,0.9,0.05,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8133086876155268
0.4,0.9,0.05,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.7985212569316081
0.4,0.9,0.05,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.800369685767098
0.4,0.9,0.05,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8299445471349353
0.4,0.9,0.1,Qwen__Qwen3-8B,prompt_level_strict_acc,0.822550831792976
0.4,0.9,0.1,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.822550831792976
0.4,0.9,0.1,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8262476894639557
0.4,0.9,0.1,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7966728280961183
0.4,0.9,0.1,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8114602587800369
0.4,0.9,0.2,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8170055452865065
0.4,0.9,0.2,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8151571164510166
0.4,0.9,0.2,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8114602587800369
0.4,0.9,0.2,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8114602587800369
0.4,0.9,0.2,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8410351201478743
0.4,1.0,0.0,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8188539741219963
0.4,1.0,0.0,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8151571164510166
0.4,1.0,0.0,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.800369685767098
0.4,1.0,0.0,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8151571164510166
0.4,1.0,0.0,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.844731977818854
0.4,1.0,0.05,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8096118299445472
0.4,1.0,0.05,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8262476894639557
0.4,1.0,0.05,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8077634011090573
0.4,1.0,0.05,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7966728280961183
0.4,1.0,0.05,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8317929759704251
0.4,1.0,0.1,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8188539741219963
0.4,1.0,0.1,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8373382624768947
0.4,1.0,0.1,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8207024029574861
0.4,1.0,0.1,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.800369685767098
0.4,1.0,0.1,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8299445471349353
0.4,1.0,0.2,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8151571164510166
0.4,1.0,0.2,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8096118299445472
0.4,1.0,0.2,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8170055452865065
0.4,1.0,0.2,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8077634011090573
0.4,1.0,0.2,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8373382624768947
0.6,0.8,0.0,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8299445471349353
0.6,0.8,0.0,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8040665434380776
0.6,0.8,0.0,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8207024029574861
0.6,0.8,0.0,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7966728280961183
0.6,0.8,0.0,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8243992606284658
0.6,0.8,0.05,Qwen__Qwen3-8B,prompt_level_strict_acc,0.833641404805915
0.6,0.8,0.05,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8022181146025879
0.6,0.8,0.05,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8188539741219963
0.6,0.8,0.05,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8114602587800369
0.6,0.8,0.05,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8170055452865065
0.6,0.8,0.1,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8262476894639557
0.6,0.8,0.1,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8096118299445472
0.6,0.8,0.1,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8188539741219963
0.6,0.8,0.1,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8040665434380776
0.6,0.8,0.1,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8280961182994455
0.6,0.8,0.2,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8262476894639557
0.6,0.8,0.2,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.7929759704251387
0.6,0.8,0.2,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8170055452865065
0.6,0.8,0.2,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7985212569316081
0.6,0.8,0.2,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8299445471349353
0.6,0.9,0.0,Qwen__Qwen3-8B,prompt_level_strict_acc,0.833641404805915
0.6,0.9,0.0,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8059149722735675
0.6,0.9,0.0,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8262476894639557
0.6,0.9,0.0,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8096118299445472
0.6,0.9,0.0,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8299445471349353
0.6,0.9,0.05,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8317929759704251
0.6,0.9,0.05,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8059149722735675
0.6,0.9,0.05,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8188539741219963
0.6,0.9,0.05,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8059149722735675
0.6,0.9,0.05,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.833641404805915
0.6,0.9,0.1,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8188539741219963
0.6,0.9,0.1,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8133086876155268
0.6,0.9,0.1,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8299445471349353
0.6,0.9,0.1,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8133086876155268
0.6,0.9,0.1,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8354898336414048
0.6,0.9,0.2,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8299445471349353
0.6,0.9,0.2,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.800369685767098
0.6,0.9,0.2,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8077634011090573
0.6,0.9,0.2,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7911275415896488
0.6,0.9,0.2,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8243992606284658
0.6,1.0,0.0,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8096118299445472
0.6,1.0,0.0,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.800369685767098
0.6,1.0,0.0,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8170055452865065
0.6,1.0,0.0,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8040665434380776
0.6,1.0,0.0,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8317929759704251
0.6,1.0,0.05,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8133086876155268
0.6,1.0,0.05,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8077634011090573
0.6,1.0,0.05,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8151571164510166
0.6,1.0,0.05,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.800369685767098
0.6,1.0,0.05,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8280961182994455
0.6,1.0,0.1,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8391866913123844
0.6,1.0,0.1,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8170055452865065
0.6,1.0,0.1,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.822550831792976
0.6,1.0,0.1,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7874306839186691
0.6,1.0,0.1,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8114602587800369
0.6,1.0,0.2,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8170055452865065
0.6,1.0,0.2,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.7966728280961183
0.6,1.0,0.2,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8133086876155268
0.6,1.0,0.2,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8096118299445472
0.6,1.0,0.2,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8280961182994455
0.8,0.8,0.0,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8151571164510166
0.8,0.8,0.0,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8280961182994455
0.8,0.8,0.0,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8188539741219963
0.8,0.8,0.0,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7874306839186691
0.8,0.8,0.0,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8410351201478743
0.8,0.8,0.05,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8262476894639557
0.8,0.8,0.05,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8059149722735675
0.8,0.8,0.05,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8207024029574861
0.8,0.8,0.05,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.789279112754159
0.8,0.8,0.05,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8280961182994455
0.8,0.8,0.1,Qwen__Qwen3-8B,prompt_level_strict_acc,0.822550831792976
0.8,0.8,0.1,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8299445471349353
0.8,0.8,0.1,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8133086876155268
0.8,0.8,0.1,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7855822550831792
0.8,0.8,0.1,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.833641404805915
0.8,0.8,0.2,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8262476894639557
0.8,0.8,0.2,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8188539741219963
0.8,0.8,0.2,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8040665434380776
0.8,0.8,0.2,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8059149722735675
0.8,0.8,0.2,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.833641404805915
0.8,0.9,0.0,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8243992606284658
0.8,0.9,0.0,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.7985212569316081
0.8,0.9,0.0,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8188539741219963
0.8,0.9,0.0,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.800369685767098
0.8,0.9,0.0,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8280961182994455
0.8,0.9,0.05,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8151571164510166
0.8,0.9,0.05,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8170055452865065
0.8,0.9,0.05,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8170055452865065
0.8,0.9,0.05,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.789279112754159
0.8,0.9,0.05,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.833641404805915
0.8,0.9,0.1,Qwen__Qwen3-8B,prompt_level_strict_acc,0.822550831792976
0.8,0.9,0.1,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.7948243992606284
0.8,0.9,0.1,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8262476894639557
0.8,0.9,0.1,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8059149722735675
0.8,0.9,0.1,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8299445471349353
0.8,0.9,0.2,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8243992606284658
0.8,0.9,0.2,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8059149722735675
0.8,0.9,0.2,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8188539741219963
0.8,0.9,0.2,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7929759704251387
0.8,0.9,0.2,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.833641404805915
0.8,1.0,0.0,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8354898336414048
0.8,1.0,0.0,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.7985212569316081
0.8,1.0,0.0,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8188539741219963
0.8,1.0,0.0,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8059149722735675
0.8,1.0,0.0,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8280961182994455
0.8,1.0,0.05,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8410351201478743
0.8,1.0,0.05,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8133086876155268
0.8,1.0,0.05,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8188539741219963
0.8,1.0,0.05,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7818853974121996
0.8,1.0,0.05,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.844731977818854
0.8,1.0,0.1,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8188539741219963
0.8,1.0,0.1,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8040665434380776
0.8,1.0,0.1,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8096118299445472
0.8,1.0,0.1,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7948243992606284
0.8,1.0,0.1,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8299445471349353
0.8,1.0,0.2,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8262476894639557
0.8,1.0,0.2,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8151571164510166
0.8,1.0,0.2,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8262476894639557
0.8,1.0,0.2,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7874306839186691
0.8,1.0,0.2,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.833641404805915
1.0,0.8,0.0,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8207024029574861
1.0,0.8,0.0,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8188539741219963
1.0,0.8,0.0,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8133086876155268
1.0,0.8,0.0,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8022181146025879
1.0,0.8,0.0,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8262476894639557
1.0,0.8,0.05,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8170055452865065
1.0,0.8,0.05,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8077634011090573
1.0,0.8,0.05,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8022181146025879
1.0,0.8,0.05,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.789279112754159
1.0,0.8,0.05,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8133086876155268
1.0,0.8,0.1,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8262476894639557
1.0,0.8,0.1,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.822550831792976
1.0,0.8,0.1,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8040665434380776
1.0,0.8,0.1,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7966728280961183
1.0,0.8,0.1,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8280961182994455
1.0,0.8,0.2,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8151571164510166
1.0,0.8,0.2,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8262476894639557
1.0,0.8,0.2,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8059149722735675
1.0,0.8,0.2,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.800369685767098
1.0,0.8,0.2,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8317929759704251
1.0,0.9,0.0,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8133086876155268
1.0,0.9,0.0,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8262476894639557
1.0,0.9,0.0,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.7985212569316081
1.0,0.9,0.0,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8040665434380776
1.0,0.9,0.0,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8151571164510166
1.0,0.9,0.05,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8151571164510166
1.0,0.9,0.05,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8170055452865065
1.0,0.9,0.05,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8151571164510166
1.0,0.9,0.05,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7763401109057301
1.0,0.9,0.05,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8188539741219963
1.0,0.9,0.1,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8354898336414048
1.0,0.9,0.1,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.822550831792976
1.0,0.9,0.1,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8040665434380776
1.0,0.9,0.1,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.8022181146025879
1.0,0.9,0.1,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8317929759704251
1.0,0.9,0.2,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8280961182994455
1.0,0.9,0.2,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8207024029574861
1.0,0.9,0.2,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8096118299445472
1.0,0.9,0.2,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7985212569316081
1.0,0.9,0.2,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.822550831792976
1.0,1.0,0.0,Qwen__Qwen3-8B,prompt_level_strict_acc,0.800369685767098
1.0,1.0,0.0,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8077634011090573
1.0,1.0,0.0,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8151571164510166
1.0,1.0,0.0,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7966728280961183
1.0,1.0,0.0,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8354898336414048
1.0,1.0,0.05,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8207024029574861
1.0,1.0,0.05,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8077634011090573
1.0,1.0,0.05,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8151571164510166
1.0,1.0,0.05,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7800369685767098
1.0,1.0,0.05,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8188539741219963
1.0,1.0,0.1,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8299445471349353
1.0,1.0,0.1,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8114602587800369
1.0,1.0,0.1,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8059149722735675
1.0,1.0,0.1,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7874306839186691
1.0,1.0,0.1,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.844731977818854
1.0,1.0,0.2,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8077634011090573
1.0,1.0,0.2,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8207024029574861
1.0,1.0,0.2,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8114602587800369
1.0,1.0,0.2,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7985212569316081
1.0,1.0,0.2,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.822550831792976
0.0,1.0,0.0,unsloth__Qwen3-8B-unsloth-bnb-4bit,prompt_level_strict_acc,0.8040665434380776
0.0,1.0,0.0,Qwen__Qwen3-8B,prompt_level_strict_acc,0.8040665434380776
0.0,1.0,0.0,Qwen__Qwen3-8B-AWQ,prompt_level_strict_acc,0.8077634011090573
0.0,1.0,0.0,Qwen3-8B-Q2_K.gguf,prompt_level_strict_acc,0.7985212569316081
0.0,1.0,0.0,Qwen3-8B-Q4_K_M.gguf,prompt_level_strict_acc,0.8262476894639557